## Using SpatialData

In [1]:
import spatialdata as sd

sdata = sd.SpatialData.read("../data/xenium_sample.zarr")

/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Access the exact pixel masks (these are usually xarray DataTrees or DataArrays)
# Note: The exact dictionary keys might be named slightly differently depending on the parser
cell_mask = sdata.labels["cell_labels"]
nucleus_mask = sdata.labels["nucleus_labels"]

print(cell_mask)

<xarray.DataTree>
Group: /
├── Group: /scale0
│       Dimensions:  (y: 17098, x: 51187)
│       Coordinates:
│         * y        (y) float64 137kB 0.5 1.5 2.5 3.5 ... 1.71e+04 1.71e+04 1.71e+04
│         * x        (x) float64 409kB 0.5 1.5 2.5 3.5 ... 5.118e+04 5.119e+04 5.119e+04
│       Data variables:
│           image    (y, x) uint32 4GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
├── Group: /scale1
│       Dimensions:  (y: 8549, x: 25593)
│       Coordinates:
│         * y        (y) float64 68kB 1.0 3.0 5.0 7.0 ... 1.709e+04 1.71e+04 1.71e+04
│         * x        (x) float64 205kB 1.0 3.0 5.0 7.0 ... 5.118e+04 5.118e+04 5.119e+04
│       Data variables:
│           image    (y, x) uint32 875MB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
├── Group: /scale2
│       Dimensions:  (y: 4274, x: 12796)
│       Coordinates:
│         * y        (y) float64 34kB 2.0 6.001 10.0 ... 1.709e+04 1.709e+04 1.71e+04
│         * x        (x) float64 102kB 2.0 6.0 10.0 ... 5.118

## Visualize Cell Boundaries Using Full Vertices in Napari

In [ ]:
import spatialdata as sd
from napari_spatialdata import Interactive

# 1. Load your spatialdata object (if not already loaded)
sdata = sd.SpatialData.read("../data/xenium_sample_fixed.zarr")

# 2. Launch the interactive napari viewer
interactive = Interactive(sdata)

2026-09-06 18:10:31.914 | WARNING  | napari_spatialdata._viewer:__init__:56 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "

ERROR: Invoking <bound method SceneCanvas.on_draw of <SceneCanvas (PyQt5) at 0x7f2ca8484e10>> repeat 2
Traceback (most recent call last):
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/OpenGL/latebind.py", line 43, in __call__
    return self._finalCall( *args, **named )
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not callable

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/vispy/app/backends/_qt.py", line 599, in wheelEvent
    vispy_event = self._vispy_canvas.events.mouse_wheel(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/vispy/util/event.py", lin

In [ ]:
# If running in a standard script (not a Jupyter notebook), keep the window open:
import napari
napari.run()

## Make Spatial data

To load a Xenium output folder into a SpatialData object (sdata), fix the index formatting, and include the full pixel-perfect raster masks, use the spatialdata-io package.

Run the following Python script to read the raw Xenium folder, process the indices, and save it as a Zarr store:

In [7]:
import spatialdata as sd
from spatialdata_io import xenium

# 1. Point to your raw Xenium output directory
xenium_folder = "/home/duydao/Projects/xenium_lung/data/Xenium_V1_humanLung_Cancer_FFPE_outs"

# 2. Read the Xenium folder into a SpatialData object 
# (This automatically loads the full-resolution pixel masks into sdata.labels 
# and the simplified polygons into sdata.shapes)
sdata = xenium(xenium_folder)

# 3. Fix the string-based index issue for both cell and nucleus boundaries
for shape_name in ["cell_boundaries", "nucleus_boundaries"]:
    if shape_name in sdata.shapes:
        boundaries = sdata.shapes[shape_name]
        # Preserve original barcodes in a metadata column
        boundaries["cell_id_str"] = boundaries.index.astype(str)
        # Convert index to standard integers (0, 1, 2...) for napari compatibility
        sdata.shapes[shape_name] = boundaries.reset_index(drop=True)

# 4. Save the corrected SpatialData object to a Zarr store
output_zarr_path = "../data/xenium_sample.zarr"
sdata.write(output_zarr_path, overwrite=True)

print(f"SpatialData object successfully created and saved to {output_zarr_path}")

/home/duydao/micromamba/envs/napari-env/lib/python3.11/site-packages/spatialdata_io/_utils.py:59: UserWarning: Nucleus boundaries are not supported for this Xenium format version (v2.0.0 early builds without label_id in the parquet). The parquet merges multinucleate cells into degenerate polygons. Skipping nucleus boundaries. You can derive nucleus polygons from the raster labels using spatialdata.to_polygons(). See https://github.com/scverse/spatialdata-io/discussions/387 for details.
  return f(*args, **kwargs)
/home/duydao/micromamba/envs/napari-env/lib/python3.11/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/home/duydao/micromamba/envs/napari-env/lib/python3.11/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/home/duydao/micromamba/envs/napari-env/lib/python3.11/contextlib.py:137: UserWarning: zarr v3 autosharding will be the def

SpatialData object successfully created and saved to data/xenium_sample.zarr
